# Transfer Learning - Classificação de Gatos vs Cachorros

Este notebook demonstra a aplicação de **Transfer Learning** para classificar imagens de gatos e cachorros usando TensorFlow.

## 📚 O que você vai aprender:

1. Como carregar e preprocessar o dataset cats_vs_dogs
2. Como usar modelos pré-treinados (MobileNetV2)
3. Como aplicar Transfer Learning
4. Como treinar e avaliar o modelo
5. Como fazer predições em novas imagens

## 1. Importar Bibliotecas

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

## 2. Configurações

In [ ]:
# Hiperparâmetros
IMG_SIZE = 160
BATCH_SIZE = 32
EPOCHS = 10
AUTOTUNE = tf.data.AUTOTUNE

CLASS_NAMES = ['Gato', 'Cachorro']

## 3. Carregar Dataset

Vamos usar o dataset `cats_vs_dogs` do TensorFlow Datasets.

In [ ]:
# Carregar dataset
(train_ds, validation_ds, test_ds), info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
)

print(f"Total de imagens: {info.splits['train'].num_examples}")
print(f"Número de classes: {info.features['label'].num_classes}")

## 4. Visualizar Amostras do Dataset

In [ ]:
plt.figure(figsize=(15, 10))

for i, (image, label) in enumerate(train_ds.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image.numpy())
    plt.title(CLASS_NAMES[label.numpy()])
    plt.axis('off')

plt.tight_layout()
plt.show()

## 5. Preprocessamento de Dados

In [ ]:
def preprocess_image(image, label):
    """Redimensiona e normaliza a imagem."""
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def augment_image(image, label):
    """Aplica data augmentation."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.2)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    return image, label

# Preparar datasets
train_dataset = (
    train_ds
    .cache()
    .shuffle(1000)
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .map(augment_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

validation_dataset = (
    validation_ds
    .cache()
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_dataset = (
    test_ds
    .cache()
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("✅ Datasets preparados!")

## 6. Visualizar Imagens Preprocessadas

In [ ]:
plt.figure(figsize=(15, 10))

for images, labels in train_dataset.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(CLASS_NAMES[int(labels[i].numpy())])
        plt.axis('off')

plt.tight_layout()
plt.show()

## 7. Criar Modelo com Transfer Learning

Vamos usar **MobileNetV2** pré-treinado no ImageNet como base.

In [ ]:
# Carregar modelo base pré-treinado
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,  # Não incluir camadas de classificação
    weights='imagenet'  # Usar pesos do ImageNet
)

# Congelar o modelo base (não treinar suas camadas)
base_model.trainable = False

print(f"Modelo base: {base_model.name}")
print(f"Número de camadas: {len(base_model.layers)}")

In [ ]:
# Criar modelo completo
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compilar modelo
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 8. Treinar o Modelo

In [ ]:
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=validation_dataset,
    verbose=1
)

## 9. Visualizar Histórico de Treinamento

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))

# Acurácia
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Treino')
plt.plot(epochs_range, val_acc, label='Validação')
plt.legend(loc='lower right')
plt.title('Acurácia')
plt.xlabel('Época')
plt.ylabel('Acurácia')

# Perda
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Treino')
plt.plot(epochs_range, val_loss, label='Validação')
plt.legend(loc='upper right')
plt.title('Perda')
plt.xlabel('Época')
plt.ylabel('Perda')

plt.tight_layout()
plt.show()

## 10. Avaliar no Conjunto de Teste

In [ ]:
loss, accuracy = model.evaluate(test_dataset)

print(f"\n📊 Resultados no conjunto de teste:")
print(f"   - Perda: {loss:.4f}")
print(f"   - Acurácia: {accuracy:.4f} ({accuracy*100:.2f}%)")

## 11. Fazer Predições

In [ ]:
plt.figure(figsize=(18, 12))

for images, labels in test_dataset.take(1):
    predictions = model.predict(images)
    
    for i in range(min(12, len(images))):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy())
        
        true_label = int(labels[i].numpy())
        pred_label = int(predictions[i] > 0.5)
        confidence = predictions[i][0] if pred_label == 1 else 1 - predictions[i][0]
        
        true_class = CLASS_NAMES[true_label]
        pred_class = CLASS_NAMES[pred_label]
        
        color = 'green' if true_label == pred_label else 'red'
        
        plt.title(f'Real: {true_class}\nPred: {pred_class} ({confidence*100:.1f}%)',
                 color=color)
        plt.axis('off')

plt.tight_layout()
plt.show()

## 12. Salvar o Modelo

In [ ]:
import os

os.makedirs('../models', exist_ok=True)
model.save('../models/cats_vs_dogs_model.h5')
print("✅ Modelo salvo em ../models/cats_vs_dogs_model.h5")

## 13. Experimento: Fine-tuning

Agora vamos descongelar algumas camadas do modelo base e treinar novamente com learning rate menor.

In [ ]:
# Descongelar as últimas 20 camadas do modelo base
base_model.trainable = True

print(f"Número de camadas no modelo base: {len(base_model.layers)}")

# Congelar todas exceto as últimas 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Recompilar com learning rate menor
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),  # 10x menor
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\n✅ Modelo preparado para fine-tuning")
print(f"Camadas treináveis: {len([l for l in model.layers if l.trainable])}")

In [ ]:
# Treinar mais algumas épocas
history_fine = model.fit(
    train_dataset,
    epochs=5,
    validation_data=validation_dataset,
    verbose=1
)

In [ ]:
# Avaliar novamente
loss, accuracy = model.evaluate(test_dataset)

print(f"\n📊 Resultados após fine-tuning:")
print(f"   - Perda: {loss:.4f}")
print(f"   - Acurácia: {accuracy:.4f} ({accuracy*100:.2f}%)")

## 14. Conclusão

Neste notebook, você aprendeu:

✅ Como carregar e preprocessar dados do TensorFlow Datasets

✅ Como aplicar Transfer Learning usando MobileNetV2

✅ Como treinar um modelo de classificação binária

✅ Como avaliar e visualizar resultados

✅ Como fazer fine-tuning para melhorar a performance

### Próximos Passos:

1. Experimente outros modelos base (ResNet50, EfficientNet, etc.)
2. Ajuste hiperparâmetros (learning rate, batch size, epochs)
3. Implemente callbacks (EarlyStopping, ReduceLROnPlateau)
4. Use o modelo para fazer predições em suas próprias imagens